# Notes for ABAD Legwheel dev

## env setup

In [1]:
# !/usr/bin/env python3
# Author: Star
from legwheel.config import RobotParams
from legwheel.models.corgi_leg import CorgiLegKinematics
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# create single leg object for testing
corgi_leg = CorgiLegKinematics(0)  # Leg index 0-3 for FL, FR, RL, RR


## Transform

In [3]:

# get transformation matrixs
# R_L_to_M: Leg fram
# e to Module frame
# R_M_to_R: Module frame to Robot frame
T_L_to_M, T_M_to_R = corgi_leg._get_transformation_matrices()


In [4]:
# vertify transformation matrices are correct by transforming a point from Leg frame to Robot frame
x_L = np.array([1, 0, 0, 1])  # x-axis unit vector in Leg frame
x_M = T_L_to_M @ x_L  # Transform to Module frame
print("x-axis unit vector in Module frame:", x_M)
y_L = np.array([0, 1, 0, 1])  # y-axis unit vector in Leg frame
y_M = T_L_to_M @ y_L  # Transform to Module frame
print("y-axis unit vector in Module frame:", y_M)
z_L = np.array([0, 0, 1, 1])  # z-axis unit vector in Leg frame
z_M = T_L_to_M @ z_L  # Transform to Module frame
print("z-axis unit vector in Module frame:", z_M)

x-axis unit vector in Module frame: [ 0.091675  0.       -1.        1.      ]
y-axis unit vector in Module frame: [0.091675 1.       0.       1.      ]
z-axis unit vector in Module frame: [1.091675 0.       0.       1.      ]


In [5]:
# verify the transformation from Module frame to Robot frame
x_M = np.array([1, 0, 0, 1])  # x-axis unit vector in Module frame
x_R = T_M_to_R @ x_M  # Transform to Robot frame
print("x-axis unit vector in Robot frame:", x_R)
y_M = np.array([0, 1, 0, 1])  # y-axis unit vector in Module frame
y_R = T_M_to_R @ y_M  # Transform to Robot frame
print("y-axis unit vector in Robot frame:", y_R)
z_M = np.array([0, 0, 1, 1])  # z-axis unit vector in Module frame
z_R = T_M_to_R @ z_M  # Transform to Robot frame
print("z-axis unit vector in Robot frame:", z_R)

x-axis unit vector in Robot frame: [0.255    1.12     0.057166 1.      ]
y-axis unit vector in Robot frame: [0.255    0.12     1.057166 1.      ]
z-axis unit vector in Robot frame: [1.255    0.12     0.057166 1.      ]


## FK & IK


### Forward Kinematics

In [6]:
q = [np.deg2rad(100), 0, 0.5]  # example joint angles [theta, beta, gamma]
foot_pos_L = corgi_leg.forward_kinematics(*q)  # foot position in Body frame
print("Foot position in Body frame:", foot_pos_L)



Foot position in Body frame: [ 0.255       0.33594511 -0.14690043]


### Inverse Kinematic

In [7]:
q = [np.deg2rad(100), np.deg2rad(10), np.deg2rad(30)]  # example joint angles [theta, beta, gamma]
foot_pos = corgi_leg.forward_kinematics(*q)  # foot position in Body frame
q_guess = q +  + np.deg2rad(30)   # initial guess for IK (perturbed from true angles)
q_inv = corgi_leg.inverse_kinematics(target_pos = foot_pos, guess_q=q_guess)  # joint angles from foot position
print("Original joint angles (rad):", q)
print("Foot position (m):", foot_pos)
print("Inverse kinematics joint angles (rad):", q_inv)

Original joint angles (rad): [np.float64(1.7453292519943295), np.float64(0.17453292519943295), np.float64(0.5235987755982988)]
Foot position (m): [ 0.20592447  0.33855347 -0.13802972]
Inverse kinematics joint angles (rad): [1.74532925 0.17453293 0.52359878]


## Trajectory

In [9]:
# motion planning while stance phase
# trying to not rolling while moving theta & gamma joints
q = [np.deg2rad(100), np.deg2rad(10), np.deg2rad(10)]  # example joint angles [theta, beta, gamma]
print(f"Original joint angles [deg]: {np.rad2deg(q)}")

# get the current rim point (alpha, w) for the foot position
geom = corgi_leg.foot_rim_contact_fk(*q)
print("Current rim point [alpha, w]:", np.rad2deg(geom))

# must know the current foot position in Body frame
foot_pos = corgi_leg.forward_kinematics(*q, *geom)  # foot position in Body frame
print("Current foot position in Body frame:", foot_pos)

# moving foot point while maintain relationship between v = [z*cos(beta),y,-z]

# Compute the velocity vector v in Body frame
# starting in Leg frame along beta direction
v_L = np.array([np.sin(q[1]), -np.cos(q[1]), 0])  # velocity in Leg frame along beta direction 
v = corgi_leg._transform_to_body(v_L, type="vec")  # transform to Body frame
print("Velocity vector v in Body frame:", v)

# check IK for a small step in the direction of v
delta = 0.01  # small step size
target_pos = foot_pos + delta * v / np.linalg.norm(v)  # target foot position after moving in the direction of v
print("Target foot position in Body frame:", target_pos)
q_guess = q
q_inv = corgi_leg.inverse_kinematics(target_pos=target_pos, guess_q=q_guess, rim_point=geom)  # joint angles from target foot position
print("Inverse kinematics joint angles for target position (deg):", np.rad2deg(q_inv))

Original joint angles [deg]: [100.  10.  10.]
Current rim point [alpha, w]: [-10.   0.]
Current foot position in Body frame: [ 0.20632806  0.25862447 -0.20107713]
Velocity vector v in Body frame: [-0.17364818  0.         -0.98480775]
Target foot position in Body frame: [ 0.20459158  0.25862447 -0.2109252 ]
Inverse kinematics joint angles for target position (deg): [104.88822734  10.02348932   9.65957505]
